In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

from pathlib import Path
import shutil
import sys
import json

import ase.io
from ase.visualize import view
import numpy as onp

from msmjax.calculators import set_up_msm_params, MSMParams
import msmjax

parentpath_charged_lj_utils = str(
    Path(msmjax.__file__).resolve().parents[2] / "examples" / "charged_lj_md"
)
sys.path.append(parentpath_charged_lj_utils)
path_charged_lj_utils = parentpath_charged_lj_utils / "utils_charged_lj"

from utils_charged_lj.energy_model import SIGMA_ANGSTROM
from utils_charged_lj.helpers import read_logfile_ase, make_md_command

# Get target pressure for NPT simulation from NVT warmup simulation

In [2]:
BASEINDIR = Path("NVT/")

atoms_init = ase.io.read(BASEINDIR / "out" / "md.traj", index=-1)
atoms_init.wrap()
sidelength_init = atoms_init.cell.cellpar()[0]
cell_init = onp.asarray(atoms_init.get_cell())
msm_params_init = MSMParams.load_json(BASEINDIR / "msm_params.json")

In [3]:
view(atoms_init)

<Popen: returncode: None args: ['/home/fbuchner/miniconda3/envs/msmjax_diron...>

In [4]:
# Get a reasonable pressure for the NPT simulation by averaging over
# the second half of the NVT trajectory:
logfile = BASEINDIR / "out" / "md.log"
loaded_log = read_logfile_ase(logfile, stress=True)
target_pressure_GPa = -loaded_log["stress_GPa"][
    len(loaded_log["stress_GPa"]) // 2 :, :3
].mean()

# Set up MSM parameters with a "safety margin" built in for cell compression/expansion

The "safe limits" for compression and expansion are chosen somewhat arbitrarily at this point.

-> It is a good idea to monitor during/after the simulation if the cell stayed within these limits.

In [5]:
# Choosing the maximum compression as twice the short-range cutoff is based on
# mere convenience, because that is the smallest cell in which the short-range
# contribution can be evaluated using the minimum-image convention without
# creating a supercell.
sidelength_at_max_compression = 2 * msm_params_init.cutoffs[0]
sidelength_at_max_expansion = 1.5 * sidelength_init

# sidelength_at_max_compression, sidelength_at_max_expansion
print(
    f'Side length in most compressed "safe" state: {sidelength_at_max_compression:.2f} Å'
)
print(
    f'Side length in most expanded "safe" state: {sidelength_at_max_expansion:.2f} Å'
)

Side length in most compressed "safe" state: 26.76 Å
Side length in most expanded "safe" state: 52.97 Å


In [6]:
cell_at_max_compression = cell_init * (
    sidelength_at_max_compression / sidelength_init
)
cell_at_max_expansion = cell_init * (
    sidelength_at_max_expansion / sidelength_init
)

In [ ]:
n_dim = 3
n_particles = len(atoms_init)

# As a rule of thumb, we want the grid spacing to be approximately equal to the
# average interparticle distance
# However, if that would be greater than the characteristic length of the
# Lennard-Jones potential (corresponding to a dilute system) we set the spacing
# to the Lennard-Jones characteristic length instead, as an estimate for how
# close particles can realistically get to each other in collisions.
# (This might be overly conservative, in production settings performance
# considerations might dictate a larger spacing.)

# The proper way to check for this condition is in the
# *state of greatest expansion* of the cell that we expect to encounter,
# because compression reduces the grid spacing.
avg_interparticle_distance_at_max_extension = (
    onp.linalg.det(cell_at_max_expansion) / n_particles
) ** (1 / n_dim)
target_spacing_at_max_extension = min(
    SIGMA_ANGSTROM, avg_interparticle_distance_at_max_extension
)

# Having determined the target spacing in the most expanded state expected,
# scale it to the most compressed state expected. This is because,
# to get the size of the kernel stencils right (large enough), setting up the
# MSM parameters will be done in the most compressed expected state.
target_spacing_at_max_compression = target_spacing_at_max_extension * (
    sidelength_at_max_compression / sidelength_at_max_expansion
)

print(
    f"target spacing at max extension: {target_spacing_at_max_extension:.2f} Å"
)
print(
    f"corresponding spacing at max compression: {target_spacing_at_max_compression:.2f} Å"
)

target spacing at max extension: 3.35 Å
corresponding spacing at max compression: 1.69 Å


In [8]:
msm_params = set_up_msm_params(
    cell=cell_at_max_compression,
    level_one_spacings=target_spacing_at_max_compression,
    level_zero_cutoff=msm_params_init.cutoffs[0],
    p=msm_params_init.p,
    pbc=msm_params_init.pbc,
    cell_mode=msm_params_init.cell_mode,
    dynamic_cell=True,
)

In [9]:
from msmjax.calculators import check_cutoffs_and_get_actual_spacings

for message, cll in zip(
    ["At max compression", "At reference cell shape", "At max expansion"],
    [cell_at_max_compression, cell_init, cell_at_max_expansion],
):
    sidelength = cll[0, 0]
    prefix = message + " " + f"(side length = {sidelength:.2f} Å):"
    print(f"  {prefix:<48} ", end="")
    actual_spacings = check_cutoffs_and_get_actual_spacings(cll, msm_params)
    print(f"{actual_spacings} Å")

  At max compression (side length = 26.76 Å):      

usage: ase [-h] [--version] [-T]
           {help,info,test,gui,db,run,band-structure,build,dimensionality,eos,ulm,find,nebplot,convert,reciprocal,completion,diff,exec}
           ...
ase: error: TclError: couldn't connect to display "localhost:21.0"
To get a full traceback, use: ase -T gui ...


[1.6725 1.6725 1.6725] Å
  At reference cell shape (side length = 35.31 Å): [2.20700422 2.20700422 2.20700422] Å
  At max expansion (side length = 52.97 Å):        [3.31050633 3.31050633 3.31050633] Å


# Write input files for NPT simulation

In [10]:
FILENAME_MD_SCRIPT = "run_md_ase.py"
FILENAME_INPUT_STRUCT = "initial_structure.xyz"
FILENAME_MSM_PARAMS = "msm_params.json"
DIRNAME_OUT = "out/"

In [11]:
# Get the temperature that the preceding NVT warmup simulation was run at:
with open(BASEINDIR / "md_setup_info.json", "r") as f:
    target_temp_K = json.load(f)["temp_K"]

In [ ]:
ensemble = "NPT"
rundir = Path(ensemble + BASEINDIR.parts[-1].split("NVT")[1] + "/")
simtime_ps = 3000.0
timestep_fs = 1.0
loginterval_fs = 500.0

md_setup_info = {
    "simtime_ps": simtime_ps,
    "timestep_fs": timestep_fs,
    "loginterval_fs": loginterval_fs,
    "ensemble": ensemble,
    "temp_K": target_temp_K,
    "pressure_GPa": target_pressure_GPa,
    "safe_sidelength_limits": [
        sidelength_at_max_compression,
        sidelength_at_max_expansion,
    ],
}

rundir.mkdir()

ase.io.write(rundir / FILENAME_INPUT_STRUCT, atoms_init)
msm_params.save_json(rundir / FILENAME_MSM_PARAMS, indent=2)
with open(rundir / "md_setup_info.json", "w") as f:
    json.dump(md_setup_info, f, indent=2)
shutil.copy(path_charged_lj_utils / FILENAME_MD_SCRIPT, rundir)
command = make_md_command(
    simtime_ps=simtime_ps,
    timestep_fs=timestep_fs,
    loginterval_fs=loginterval_fs,
    filename_md_script=FILENAME_MD_SCRIPT,
    filename_input_struct=FILENAME_INPUT_STRUCT,
    filename_msm_params=FILENAME_MSM_PARAMS,
    dirname_out=DIRNAME_OUT,
    ensemble=ensemble,
    temp_K=target_temp_K,
    pressure_GPa=target_pressure_GPa,
)
with open(rundir / "runscript.sh", "w") as f:
    f.write(command)